In [1]:
# ==========================================
# COMPLETE THEMATIC NEWS → RETURN PIPELINE
# ==========================================

import os
import ast
import numpy as np
import pandas as pd
import psycopg2
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# LOAD ENV + CONNECT
# ==========================================

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
openai_model = "text-embedding-3-small"

DB_CONFIG = {
    "dbname": os.getenv("DB_NAME"),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "host": os.getenv("DB_HOST"),
    "port": os.getenv("DB_PORT")
}

SYMBOL = "2222"

conn = psycopg2.connect(**DB_CONFIG)
conn.autocommit = True

print(f"Building pipeline for: {SYMBOL}")

# ==========================================
# LOAD PRICES
# ==========================================

prices_query = f"""
SELECT date, close
FROM public.prices
WHERE symbol = '{SYMBOL}'
ORDER BY date ASC;
"""

prices_df = pd.read_sql(prices_query, conn)
prices_df['target_return'] = prices_df['close'].pct_change()

# ==========================================
# LOAD NEWS EMBEDDINGS
# ==========================================

emb_query = """
SELECT news_date::date AS date, embedding
FROM embeddings
ORDER BY news_date ASC;
"""

emb_df = pd.read_sql(emb_query, conn)
emb_df['embedding'] = emb_df['embedding'].apply(ast.literal_eval)

emb_matrix = pd.DataFrame(emb_df['embedding'].tolist())

print("Total news loaded:", len(emb_df))

# ==========================================
# GENERATE KEYWORD EMBEDDINGS
# ==========================================

keywords_list = [
    "global energy markets",
    "oil production and crude prices"
]

response = client.embeddings.create(
    model=openai_model,
    input=keywords_list
)

energy_embedding = np.array(response.data[0].embedding)
oil_embedding = np.array(response.data[1].embedding)

keywords = np.vstack([energy_embedding, oil_embedding])

print("Keyword embeddings generated.")
print("Embedding dimension:", len(energy_embedding))

# ==========================================
# COSINE SIMILARITY FILTER
# ==========================================

similarities = cosine_similarity(emb_matrix, keywords)
max_similarity = similarities.max(axis=1)

emb_df['similarity'] = max_similarity

SIM_THRESHOLD = 0.30
filtered_df = emb_df[emb_df['similarity'] > SIM_THRESHOLD].copy()

print("Relevant news after filtering:", len(filtered_df))

if len(filtered_df) == 0:
    raise ValueError("No news passed similarity threshold. Lower SIM_THRESHOLD.")

# ==========================================
# DAILY AGGREGATION
# (Mean + Strongest Absolute Signal)
# ==========================================

filtered_matrix = pd.DataFrame(filtered_df['embedding'].tolist())
filtered_matrix['date'] = filtered_df['date'].values

daily_mean = filtered_matrix.groupby('date').mean()
daily_max = filtered_matrix.groupby('date').agg(lambda x: x.abs().max())

daily_mean.columns = [f"mean_{i}" for i in range(daily_mean.shape[1])]
daily_max.columns = [f"max_{i}" for i in range(daily_max.shape[1])]

daily_embeddings = pd.concat([daily_mean, daily_max], axis=1).reset_index()

print("Days with relevant news:", len(daily_embeddings))

# ==========================================
# PCA AFTER AGGREGATION
# ==========================================

feature_cols = [col for col in daily_embeddings.columns if col != "date"]

scaler_features = StandardScaler()
scaled_features = scaler_features.fit_transform(daily_embeddings[feature_cols])

pca = PCA(n_components=30)
pca_features = pca.fit_transform(scaled_features)

pca_df = pd.DataFrame(
    pca_features,
    columns=[f'pc_{i}' for i in range(30)]
)

pca_df['date'] = daily_embeddings['date'].values

# ==========================================
# MERGE WITH PRICES
# ==========================================

merged = pd.merge(prices_df, pca_df, on='date', how='inner')

# Shift features (News_{t-1} → Return_t)
pc_cols = [col for col in merged.columns if col.startswith("pc_")]

for col in pc_cols:
    merged[col] = merged[col].shift(1)

merged = merged.dropna()

X = merged[pc_cols]
y = merged['target_return']

# ==========================================
# TRAIN / TEST SPLIT (Time-based)
# ==========================================

split_index = int(len(merged) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# Scale model inputs
scaler_model = StandardScaler()
X_train_scaled = scaler_model.fit_transform(X_train)
X_test_scaled = scaler_model.transform(X_test)

# ==========================================
# RIDGE REGRESSION
# ==========================================

model = Ridge(alpha=10)
model.fit(X_train_scaled, y_train)

y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print("\n===== THEMATIC NEWS RETURN MODEL =====")
print("Train R²:", round(train_r2, 4))
print("Test R² :", round(test_r2, 4))

OperationalError: connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?


In [11]:
# LOAD PRICES
prices_query = f"""
SELECT date, open, high, low, close,
       volume, turnover, num_trades,
       change, change_pct
FROM public.prices
WHERE symbol = '{SYMBOL}'
ORDER BY date ASC;
"""

prices_df = pd.read_sql_query(prices_query, conn)
prices_df['date'] = pd.to_datetime(prices_df['date'])

# CREATE LAGS
prices_df['price_1y'] = prices_df['close'].shift(252)
prices_df['price_3m'] = prices_df['close'].shift(63)
prices_df['price_1m'] = prices_df['close'].shift(21)
prices_df['price_5d'] = prices_df['close'].shift(5)
prices_df['price_4d'] = prices_df['close'].shift(4)
prices_df['price_3d'] = prices_df['close'].shift(3)
prices_df['price_2d'] = prices_df['close'].shift(2)
prices_df['price_1d'] = prices_df['close'].shift(1)

prices_df['target_price_today'] = prices_df['close']

/tmp/ipykernel_114560/794225821.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  prices_df = pd.read_sql_query(prices_query, conn)
